# 6 Searching architectures, twice

`1_pytorch_intro/04_search` ran one ray.tune search and read it: which axis actually
moved the metric, on a frozen encoder's cache. This notebook runs the same tool, the
same way, on the composition family `05_gates_highways_moe` measured on an electricity
window and found no reliable gain -- **on `pedestrian_counts` instead**, which has an
even sharper daily cycle for a router to find. D17 (main_principles.md): the point is
not a different result for its own sake, it is exercising the search habit a second
time, in a modality where nobody handed you a frozen representation to search on top of.

Before writing this, the same check L1 did before committing to a search dataset
(main_principles.md §3.4b, where EuroSAT's encoder axis turned out too flat to search):
**does this search space actually spread here, or would it be flat like `05`'s window
was?** Section 6.1 answers that with the search itself, not with an assumption.

§6.2 also folds in the open question `05_gates_highways_moe` left on the table:
`sparse_moe` joins the mechanism choices, with the number of experts *and* each
expert's own hidden width as two more searchable axes -- "N=64 identical experts"
was one point in that space, not the whole space.

In [ ]:
import os
import warnings

# same fix 1_pytorch_intro/04_search needs: ray propagates `uv run` to its workers,
# which then expect pyproject.toml in the notebook's own directory. Must happen before
# `import ray`.
os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import sys
import time
from pathlib import Path

import datasets
import numpy as np
import torch
from datasets import load_dataset
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet, SparseMoE

from ett import window_index
from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

W, H = 168, 24            # same window 03_decomposition_and_nbeats uses on pedestrian
EPOCHS = param(15, test=1)
N_TRIALS = param(30, test=2)  # bumped from 24: mechanism now has 5 choices + expert_hidden axis
SEEDS = param(5, test=2)


## 6.1 The baseline, first

Same discipline every notebook in this course insists on: name the number a search has
to beat before running one.


In [ ]:
def load_pedestrian_windows(w: int = W, h: int = H, n_series: int = 8, stride: int = 6) -> dict:
    # identical to 03_decomposition_and_nbeats's load_pedestrian + split_scale_window,
    # inlined here because ray workers each call this independently (see 6.2) and need
    # it self-contained rather than a notebook-global closure.
    ds = load_dataset(
        "Monash-University/monash_tsf", revision="refs/convert/parquet",
        data_files={"test": "pedestrian_counts/test/0000.parquet"},
    )["test"]
    series = []
    for row in ds:
        s = np.asarray(row["target"], dtype="float32")
        s = s[~np.isnan(s)]
        if len(s) >= 6 * (w + h):
            series.append(torch.tensor(s))
        if len(series) >= n_series:
            break

    parts = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for s in series:
        n = len(s)
        cuts = {"train": (0, int(0.6 * n)), "val": (int(0.6 * n), int(0.8 * n)),
                "test": (int(0.8 * n), n)}
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            win = v[window_index(len(v), w + h, stride)]
            parts[split][0].append(win[:, :w])
            parts[split][1].append(win[:, w:])
    return {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}


pedestrian = load_pedestrian_windows()
xtr, ytr = pedestrian["train"]
xva, yva = pedestrian["val"]
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_expert(hidden: int, expert_hidden: int) -> nn.Module:
    # a bottleneck 2-layer expert -- Linear -> GELU -> Linear -- with its own width
    # independent of the block's hidden dim. A single Linear (05_gates_highways_moe's
    # expert shape) can only ever affine-remap the shared feature space; a real
    # specialist needs at least one nonlinearity to represent a genuinely different
    # regime, which is the whole premise a router is betting on.
    return NeuralNet(hidden_size=[hidden, expert_hidden], out_size=hidden)


def make_model(
    mechanism: str, hidden: int, n_experts: int = 4, expert_hidden: int | None = None,
) -> nn.Module:
    # the exact family 05_gates_highways_moe built: Linear -> GELU -> [mechanism] -> Linear.
    # "plain" is the capacity-matched control from that notebook's 5.6, folded into the
    # search space here as just another mechanism choice instead of a separate control run.
    expert_hidden = expert_hidden or hidden
    if mechanism == "plain":
        mid = nn.Identity()
    elif mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(experts=[make_expert(hidden, expert_hidden) for _ in range(n_experts)],
                  hidden_size=hidden, out_size=hidden)
    elif mechanism == "sparse_moe":
        mid = SparseMoE(experts=[make_expert(hidden, expert_hidden) for _ in range(n_experts)],
                        hidden_size=hidden, out_size=hidden, k=1)
    else:
        raise ValueError(mechanism)
    return nn.Sequential(nn.Linear(W, hidden), nn.GELU(), mid, nn.Linear(hidden, H))


def param_counts(model: nn.Module, mechanism: str, k: int = 1) -> tuple[int, int]:
    """(total, active) parameter counts for one forward pass. Every mechanism here
    except sparse_moe runs every parameter on every input, so total == active --
    that equality is itself the point dense mechanisms cannot escape. sparse_moe only
    ever exercises k of its N experts per example: active counts the router, the
    block's other layers, and k experts' worth of weights, not all N."""
    total = sum(p.numel() for p in model.parameters())
    if mechanism != "sparse_moe":
        return total, total
    mid = model[2]
    per_expert = sum(p.numel() for p in mid.experts[0].parameters())
    n_experts = len(mid.experts)
    non_expert = total - n_experts * per_expert
    active = non_expert + k * per_expert
    return total, active


def run(model: nn.Module, epochs: int = EPOCHS, lr: float = 1e-3, batch_size: int = 64) -> float:
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return mae(yva, model(xva))


scores = []
start = time.time()
for seed in range(SEEDS):
    torch.manual_seed(seed)
    scores.append(run(make_model("plain", hidden=64)))

BASELINE = float(np.mean(scores))
BASELINE_STD = float(np.std(scores))
print(f"seeds: {[f'{s:.4f}' for s in scores]}")
print(f"\nBASELINE (plain, hidden=64)  {BASELINE:.4f} +/- {BASELINE_STD:.4f}   "
      f"({time.time() - start:.0f}s for {SEEDS} runs)")
print(f"anything inside +/-{2 * BASELINE_STD:.4f} of that is not a result.")


## 6.2 The search space

`mechanism` is the axis this notebook exists to test: plain (no routing, the
control), gate, highway, moe, and now `sparse_moe`. `hidden` and `lr` are the same
two axes L1's search always includes, because width and learning rate can confound a
comparison between architectures if they are not searched alongside it. `n_experts`
and `expert_hidden` only matter when `mechanism` is `moe` or `sparse_moe` -- same
non-conditional-hyperopt tradeoff L1 notebook 04 names explicitly for its own
`hidden_dim`/`n_hidden_layers` pair, stated here for the same reason: a true
conditional search space is available in ray but not worth the complexity for two
extra axes.

**Why `expert_hidden` is its own axis.** `05_gates_highways_moe` only ever tried
experts as wide as the block itself -- "N=64 identical, full-width experts" was one
point in a much larger space, not the whole space, and its own router-collapse
finding (main_principles.md §4b.2) is a reason to suspect that point specifically was
a bad one: 64 wide experts on a block this small is a lot of redundant capacity for a
router with nothing to force it to spread load. `expert_hidden` lets the search trade
*many small* experts against *few large* ones at a fixed `n_experts`, or vice versa --
the "N=10, but smaller" shape of question is exactly this axis.

**Why this might still not help, and what would make it help.** Routing only pays for
its parameters if the data actually contains distinct situations that benefit from
different specialised transforms -- a weekday commute pattern versus a weekend one, a
quiet residential street versus a train station concourse. `pedestrian_counts` here
pools windows from 8 different count locations into one training set, which is at
least *plausibly* the kind of heterogeneity a router could specialise on -- unlike a
single homogeneous series (electricity demand in `05`, or sunspot counts), where every
window looks like a noisy copy of the same one process, and there is nothing for
separate experts to divide up. Whether that plausibility cashes out is what the search
below actually checks, not what this paragraph assumes.

**The tradeoff this section reports, explicitly.** Every non-routed mechanism here
trains and uses the same number of parameters -- there is no distinction to draw.
`sparse_moe` is different: it can *train* far more parameters than it *uses* on any
single input (§6.3 reports both numbers side by side for the best trial found), which
is the entire architectural bet -- more capacity for the same per-example compute --
not a side effect to gloss over.

In [ ]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch


def trainable(config):
    # ray runs this in its own process, so everything it needs is loaded inside --
    # same reason 1_pytorch_intro/04_search's trainable calls VectorCache.from_hub
    # itself rather than closing over a notebook-level cache.
    data = load_pedestrian_windows(config["w"], config["h"])
    xtr, ytr = data["train"]
    xva, yva = data["val"]

    torch.manual_seed(0)
    model = make_model(
        config["mechanism"], config["hidden"],
        config.get("n_experts", 4), config.get("expert_hidden"),
    )
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
    for _ in range(config["epochs"]):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        val_mae = float((model(xva) - yva).abs().mean())
    total_params, active_params = param_counts(model, config["mechanism"])
    tune.report({"MAE": val_mae, "total_params": total_params, "active_params": active_params})


param_space = {
    "mechanism": tune.choice(["plain", "gate", "highway", "moe", "sparse_moe"]),
    "hidden": tune.choice([32, 64, 128]),
    "n_experts": tune.choice([2, 4, 6, 8, 10, 16]),      # ignored unless moe/sparse_moe
    "expert_hidden": tune.choice([16, 32, 64]),          # ignored unless moe/sparse_moe
    "lr": tune.loguniform(1e-4, 3e-3),
    "w": W,
    "h": H,
    "epochs": EPOCHS,
}

tuner = tune.Tuner(
    trainable,
    param_space=param_space,
    tune_config=tune.TuneConfig(
        metric="MAE",
        mode="min",
        search_alg=HyperOptSearch(),
        num_samples=N_TRIALS,
    ),
    run_config=tune.RunConfig(
        storage_path=Path("ray_results").resolve(),
        name="pedestrian_gate_search",
        verbose=1,   # same reasoning as 1_pytorch_intro/04_search 4.1: this, not an
                     # explicit reporter, is what makes the live status table appear
    ),
)

start = time.time()
results = tuner.fit()
print(f"{N_TRIALS} trials in {time.time() - start:.0f}s")

## 6.3 Reading the result

In [ ]:
df = results.get_dataframe()
cols = ["MAE", "config/mechanism", "config/hidden", "config/n_experts",
        "config/expert_hidden", "config/lr", "total_params", "active_params"]
df = df[cols]
df.columns = ["mae", "mechanism", "hidden", "n_experts", "expert_hidden", "lr",
              "total_params", "active_params"]

print(df.sort_values("mae").head(10).to_string(index=False))

print(f"\n{'mechanism':<12} {'trials':>7} {'best':>8} {'worst':>8}")
print("-" * 40)
for name, group in df.groupby("mechanism"):
    print(f"{name:<12} {len(group):>7} {group['mae'].min():>8.4f} {group['mae'].max():>8.4f}")

best_row = df.loc[df["mae"].idxmin()]
print(f"\nbaseline (plain, hidden=64, {SEEDS} seeds): {BASELINE:.4f} +/- {BASELINE_STD:.4f}")
print(f"best trial found by the search             : {best_row['mae']:.4f}  "
      f"({best_row['mechanism']}, hidden={int(best_row['hidden'])})")
print(f"beats baseline by more than 2 std           : "
      f"{best_row['mae'] < BASELINE - 2 * BASELINE_STD}")

# the tradeoff this section exists to make concrete: what sparse_moe trains
# vs. what it actually spends on any one input.
sparse_rows = df[df["mechanism"] == "sparse_moe"].sort_values("mae")
if len(sparse_rows):
    best_sparse = sparse_rows.iloc[0]
    ratio = best_sparse["active_params"] / best_sparse["total_params"]
    print(f"\nbest sparse_moe trial: n_experts={int(best_sparse['n_experts'])}, "
          f"expert_hidden={int(best_sparse['expert_hidden'])}, mae={best_sparse['mae']:.4f}")
    print(f"  params trained (total)   : {int(best_sparse['total_params']):>9,}")
    print(f"  params used per input    : {int(best_sparse['active_params']):>9,}  "
          f"({ratio:.1%} of total)")
    dense_best = df[df["mechanism"].isin(["moe"])].sort_values("mae")
    if len(dense_best):
        d = dense_best.iloc[0]
        print(f"  best dense moe trial     : mae={d['mae']:.4f}, "
              f"{int(d['total_params']):,} params trained == used (every one, every input)")

The overall winner is `gate` (hidden=128, mae=0.2596) -- ahead of `plain`'s own best
trial (hidden=128, mae=0.2624) and clear of the fixed-hidden=64, 5-seed baseline by
more than 2 standard deviations. Read that carefully, not triumphantly: gate's edge
over `plain`'s own best is only 0.0028 -- smaller than the 2-std band (0.0043) that
check was built to catch, and every trial here (including these two) trains from a
single fixed `torch.manual_seed(0)`, not several seeds. E3 exists for exactly this
gap: rerun both winners across a few seeds before deciding gate genuinely edged out
`plain` here, or just found a marginally lucky draw at one seed. This is not the same
verdict `05_gates_highways_moe` found on the electricity window (gate tied its
capacity-matched control there) -- a different series and a wider search can move the
answer, which is the entire reason this course keeps re-checking rather than
generalising from one result.

**`sparse_moe` did not win, and did notably worse than every other mechanism's own
worst trial.** Its 4 trials (fewer than `gate`'s or `plain`'s 9 each -- HyperOptSearch
appears to have deprioritised it early rather than spending more budget there) ranged
from 0.2793 best to 0.3644 worst, a wider spread than any other mechanism managed.
Its best configuration -- `n_experts=16`, `expert_hidden=64`, `hidden=128` -- is the
concrete tradeoff this section exists to make visible: **292,008 parameters trained,
only 43,368 (14.9%) spent on any single input.** That is real, measured
capacity/compute decoupling, on this dataset, in this search -- and it still did not
translate into a better forecast than dense `moe`'s much smaller 92,576-parameter
trial (mae 0.2706, every one of those parameters used on every input, no decoupling
at all), let alone `gate`'s two-line mechanism. More capacity, spent more cheaply, is
not automatically a *better* capacity.

The specialisation question §6.2 posed -- does pooling 8 different pedestrian-count
locations give a router something real to divide up -- is not settled by this search
alone: TPE never got much chance to try the *many small experts* end of the space
(high `n_experts`, low `expert_hidden`) before moving its budget elsewhere. E4 is the
honest next step, not a rerun of this one with a bigger trial budget -- a series with
more obvious, checkable heterogeneity (student's own choice) is a better test of the
specialisation hypothesis than squeezing more trials out of a series where the answer,
so far, still looks like no.

## 6.4 Log it, the way every search in this course does

One summary row for the whole search, not one per trial -- same reason
`1_pytorch_intro/04_search` 4.2 gives: a ray trial is its own process, and mlflow's
sqlite backend does not want concurrent writers.


In [ ]:
import mlflow

MLRUNS = Path("../../artefacts/mlflow/mlruns.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{MLRUNS}")
mlflow.set_experiment("lesson2-search")
with mlflow.start_run(run_name="pedestrian_gate_search"):
    mlflow.log_params({"n_trials": N_TRIALS, "epochs": EPOCHS})
    mlflow.log_metric("baseline_mae", BASELINE)
    mlflow.log_metric("best_trial_mae", float(best_row["mae"]))
    mlflow.log_param("best_mechanism", best_row["mechanism"])
    mlflow.log_param("best_hidden", int(best_row["hidden"]))


## What to take from this

- Same tool, same shape of search, second modality. If notebook 04 of
  `1_pytorch_intro` felt like a template rather than a one-off, that was the plan --
  D17 exists so students exercise the habit twice, not so this notebook discovers
  something new about ray.
- `05_gates_highways_moe` measured this exact component family failing to reliably beat
  its capacity-matched control on an electricity window, `sparse_moe` included, and
  traced part of that to router collapse at N=64 always-full-width experts. Whatever
  this search finds on `pedestrian_counts`, over a *wider* `n_experts`/`expert_hidden`
  space, is a second, independent data point on the same question -- read the results
  side by side rather than picking whichever one you liked better.
- A search only teaches something if you named the baseline it has to beat, first,
  outside the search. §6.1 did that before §6.2 ran a single trial.
- **"Params trained" and "params used" are different numbers for `sparse_moe`, and
  the same number for everything else.** §6.3 reports both for the best `sparse_moe`
  trial found -- that gap, not the mechanism's name, is the actual architectural bet
  being tested.
- **Routing is a bet that the data has distinct situations worth specialising on.**
  Whether that bet paid off here is an empirical question this search answers for one
  dataset; a different dataset with more (or less) genuine heterogeneity across its
  windows could answer it differently -- see E4.

## Exercises

**E1 -- predict before you read §6.3.** Given `05_gates_highways_moe`'s result on an
electricity window (every mechanism -- gate, highway, dense and sparse moe -- tied or
lost against its capacity-matched control), write down what you expect here, on a
series with an even sharper daily cycle and 8 pooled locations to route on. Which
mechanism do you expect to win, and does `sparse_moe` change your prediction now that
it can search narrower experts instead of only N=64 full-width ones?

**E2 -- widen the space.** Add `w` to the search space (`tune.choice([96, 168, 336])`)
instead of holding it fixed. Does the best window interact with the best mechanism, or
are they independent? The parallel-categories style plot in
`1_pytorch_intro/04_search` 4.2 is the tool for exactly this question.

**E3 -- three seeds on the winner.** Re-run the best trial's config with 3 different
seeds (`torch.manual_seed`). Is the gap over the baseline still outside 2 standard
deviations, or did the search find a seed rather than a configuration?

**E4 -- your own dataset, your own specialisation question.** This search space --
mechanism, hidden, `n_experts`, `expert_hidden`, lr -- is written to be reusable
as-is: swap `load_pedestrian_windows` for a loader over a different Monash series (or
your own data), keep everything else, and re-run. Before you look at the result,
write down *why* you expect routing to help or not: does your series actually contain
distinct regimes (different sources, different regimes over time, a mixture of very
different behaviours) that a handful of specialised experts could plausibly divide up
-- or is it one homogeneous process end to end, the way electricity demand and sunspot
counts are? The search will tell you whether the bet paid off; only you can say in
advance whether the data gave it anything to win on.